# Day 1 — LLM Routing, Model Behaviour and Agent Design Foundations

### Learning outcomes
By the end of this practical you should be able to compare hosted and local inference, observe provider/model/latency metadata, understand why an LLM call is not yet an agent, preserve conversational state explicitly, and create a bounded agent specification for a realistic fulfilment-centre support scenario.

>**Dr Julius Sechang Mboli**
>
>**DAIM, University of Hull**
>
>**https://www.hull.ac.uk/staff-directory/julius-mboli**
>
>**https://www.linkedin.com/in/engr-julius-sechang-mboli/**
>
>**https://jsmboli.github.io/**

In [2]:
from pathlib import Path
import os, sys, json, time, importlib.util
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Locate the package without relying on a fixed working directory.
cwd = Path.cwd().resolve()
RESOURCE_DIR = None
for p in [cwd, *cwd.parents]:
    if (p / "resources" / "src").exists():
        RESOURCE_DIR = p / "resources"
        break
    if (p / "src").exists() and (p / "notebooks").exists() and (p / "data").exists():
        RESOURCE_DIR = p
        break
if RESOURCE_DIR is None:
    raise RuntimeError("Could not locate resources/src. Extract the complete bootcamp ZIP and open this notebook from inside it.")

SRC = RESOURCE_DIR / "src"
DATA = RESOURCE_DIR / "data"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from providers import ProviderRouter, ProviderError, make_messages
from notebook_utils import (
    environment_table, provider_table, response_table,
    run_with_progress, progress_indicator, display_response, display_note,
)

router = ProviderRouter(verbose=True)
display(environment_table(RESOURCE_DIR, router))

,item,value
0,Python,3.13.5
1,Kernel executable,/Users/silviyadharshini/Desktop/agentic-ai-bootcamp-resources-main/resources/.venv/bin/python
2,Working directory,/Users/silviyadharshini/Desktop/agentic-ai-bootcamp-resources-main/resources/notebooks
3,Resources directory,/Users/silviyadharshini/Desktop/agentic-ai-bootcamp-resources-main/resources
4,Data directory,/Users/silviyadharshini/Desktop/agentic-ai-bootcamp-resources-main/resources/data
5,Environment files found,/Users/silviyadharshini/Desktop/agentic-ai-bootcamp-resources-main/resources/.env


In [3]:
status = run_with_progress("Checking Groq, Ollama and optional providers", router.diagnose, True)
display(provider_table(status))

PRIMARY_PROVIDER = (
    "groq" if status["groq"]["available"]
    else "ollama" if status["ollama"]["available"]
    else "mimic"
)
print("Primary live provider for this notebook:", PRIMARY_PROVIDER)
print("Groq model:", status["groq"].get("model"))
print("Ollama model:", status["ollama"].get("model"))

,provider,available,selected_model,endpoint,diagnostic_s,detail
0,groq,False,qwen/qwen3.6-27b,https://api.groq.com/openai/v1,NaN,"GROQ_API_KEY is not set. Create a Groq API key, store it as GROQ_API_KEY, restart the kernel, and rerun Day 0."
1,ollama,False,NaN,http://127.0.0.1:11434,0.317,"Cannot reach Ollama at http://127.0.0.1:11434: ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download. Ollama CLI is not on this kernel's PATH. If `ollama list` works in Terminal but this fails in Jupyter, compare the notebook kernel, host and OLLAMA_BASE_URL."
2,openai,False,gpt-5.6-luna,,NaN,Key presence check only; Day 0 reports this as optional.
3,gemini,False,gemini-3.7-flash,,NaN,Key presence check only; Day 0 reports this as optional.
4,anthropic,False,claude-sonnet-5,,NaN,Key presence check only; Day 0 reports this as optional.
5,deepseek,False,deepseek-v4-flash,,NaN,Key presence check only; Day 0 reports this as optional.
6,mimic,True,deterministic-classroom-mimic,,NaN,Offline deterministic teaching fallback.


Primary live provider for this notebook: mimic
Groq model: qwen/qwen3.6-27b
Ollama model: None


## 1. Same business task, different providers

A useful provider abstraction lets us keep the **task constant** while changing the inference backend. This makes differences in latency, output length and operational constraints visible without rewriting the application.

We will compare every provider that is genuinely available; the offline mimic route remains a control condition rather than pretending to be a real LLM.

In [4]:
from tqdm.auto import tqdm

prompt = """
You are advising an adult learner preparing for a UK fulfilment-centre industrial visit.
Propose four bounded agentic-AI use cases across receiving, stowing, picking, packing or exception handling.
For each use case give: user, goal, data/tool, state, permitted action, prohibited action, human-review point, and one failure mode.
Do not claim access to Amazon internal systems or non-public data.
"""

records=[]
responses={}
for provider in tqdm(["groq", "ollama", "mimic"], desc="Provider comparison"):
    if provider != "mimic" and not status[provider]["available"]:
        continue
    try:
        r = run_with_progress(
            f"{provider} — bounded use-case analysis",
            router.chat, make_messages(prompt), provider=provider,
            max_tokens=650, fallback_on_error=False,
        )
        responses[provider]=r
        records.append(r.summary())
    except Exception as exc:
        records.append({"provider":provider,"model":None,"latency_s":None,"error":str(exc)})

display(pd.DataFrame(records))
for provider,r in responses.items():
    display_response(r, f"{provider.upper()} answer")

Provider comparison:   0%|          | 0/3 [00:00<?, ?it/s]

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,mimic,deterministic-classroom-mimic,0.0,None,None,None,None,None,offline,none


### MIMIC answer

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,mimic,deterministic-classroom-mimic,0.0,None,None,None,None,None,offline,none


A bounded fulfilment-centre agent can retrieve approved process guidance, summarise exceptions, calculate operational metrics and pause sensitive actions for human review.

### Discussion prompts
1. Which provider gave the fastest first useful answer?
2. Which answer was most grounded in the constraints we explicitly provided?
3. Does a larger/hosted model automatically make the workflow safer? Why not?
4. What data must stay local in a real organisation, and what could be sent to a hosted API?
5. Which measurements would you record in production: latency, tokens, errors, tool calls, human overrides, cost, groundedness?

## 2. Application-managed conversational state

Most chat APIs do not remember a previous call unless the application resends prior messages or an orchestration layer persists them. Here we make the state visible in a Python list before introducing LangGraph state tomorrow.

In [5]:
conversation = [
    {"role":"system","content":"You are a precise operations-analytics teaching assistant. Never invent access to company systems."},
    {"role":"user","content":"We are designing a packing-exception support agent. Define a safe scope in five bullets."},
]

r1 = run_with_progress(
    "Conversation turn 1", router.chat, conversation,
    provider=PRIMARY_PROVIDER, max_tokens=350,
    fallback_on_error=(PRIMARY_PROVIDER=="mimic")
)
display_response(r1, "Turn 1")
conversation += [{"role":"assistant","content":r1.text}, {
    "role":"user","content":"Now add three tools, state fields, an audit log and one human approval gate without expanding the scope."
}]
r2 = run_with_progress(
    "Conversation turn 2", router.chat, conversation,
    provider=PRIMARY_PROVIDER, max_tokens=450,
    fallback_on_error=(PRIMARY_PROVIDER=="mimic")
)
display_response(r2, "Turn 2")
print("Messages currently held by the application:", len(conversation)+1)

### Turn 1

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,mimic,deterministic-classroom-mimic,0.0,None,None,None,None,None,offline,none


Agentic AI combines state, tools, model decisions, observations and controlled actions. Use deterministic workflow steps where reliability matters and LLM decisions where flexibility adds value.

### Turn 2

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,mimic,deterministic-classroom-mimic,0.0,None,None,None,None,None,offline,none


Pause at a human-in-the-loop checkpoint and present the proposed action, evidence, risk, and approve/edit/reject options.

Messages currently held by the application: 5


## 3. Structured agent card

An agent design should be explicit enough to review before implementation. The following schema separates **goal**, **tools**, **state**, **authority**, **human oversight** and **failure handling**.

In [7]:
structured_prompt = """
Return JSON only with this schema:
{
  "agent_name": string,
  "primary_user": string,
  "goal": string,
  "inputs": [string],
  "tools": [{"name": string, "purpose": string}],
  "state_fields": [string],
  "allowed_actions": [string],
  "prohibited_actions": [string],
  "human_review_triggers": [string],
  "audit_events": [string],
  "failure_modes": [string]
}
Design a bounded packing-exception support agent for a teaching simulation.
"""
r = run_with_progress(
    "Generating structured agent card", router.chat, make_messages(structured_prompt),
    provider=PRIMARY_PROVIDER, max_tokens=650, fallback_on_error=(PRIMARY_PROVIDER=="mimic")
)
display_response(r, "Structured agent-card generation")

# Parse only when the provider really returned JSON.
try:
    card=json.loads(r.text)
    display(pd.json_normalize(card, sep="."))
except Exception as exc:
    print("JSON validation result: response was not strict JSON ->", exc)
    print("This is a useful reason to use structured-output features or schema validation in production.")

### Structured agent-card generation

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,mimic,deterministic-classroom-mimic,0.0,None,None,None,None,None,offline,none


Pause at a human-in-the-loop checkpoint and present the proposed action, evidence, risk, and approve/edit/reject options.

JSON validation result: response was not strict JSON -> Expecting value: line 1 column 1 (char 0)
This is a useful reason to use structured-output features or schema validation in production.


## 4. From LLM to agent

An **LLM call** maps messages to a response. An **agentic system** adds state, tools, observations, iteration, authority boundaries, persistence, and often human intervention. Tomorrow we implement those components with LangGraph rather than merely describing them in a prompt.

**Preparation:** sketch the state dictionary your agent would need after each step and identify which steps should be deterministic rather than decided by the LLM.